# Install Library

In [ ]:
!pip install transformers
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 12.9 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.
torch 2.5.1+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 whi

# Import Library

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer
from google.colab import drive

# Connect to Drive

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Path folder tempat file CSV berada
folder_path = "/content/drive/MyDrive/File/CODES/Summarize/2_Summary_Detik_Hot_50_Base1/Sum_Detik_Hot_50_Base1"

# Mengambil semua file dengan ekstensi .csv dalam folder
csv_files = [file for file in os.listdir(folder_path) if file.endswith(".csv")]

# Membaca dan menggabungkan semua file
dataframes = []
for file_name in csv_files:
    file_path = os.path.join(folder_path, file_name)
    df = pd.read_csv(file_path)
    dataframes.append(df)

# Menggabungkan semua DataFrame
combined_df = pd.concat(dataframes, ignore_index=True)

In [ ]:
# Menyimpan hasil penggabungan ke file baru
output_path = os.path.join(folder_path, "/content/drive/MyDrive/File/CODES/Similarity/2_Similarity_Detik_Hot_Base1/detikhot_sum_50_base1_concat.csv")
combined_df.to_csv(output_path, index=False)

In [ ]:
combined_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2063 entries, 0 to 2062
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   title         2063 non-null   object
 1   category      2063 non-null   object
 2   publish_date  2063 non-null   object
 3   author        2063 non-null   object
 4   article_url   2063 non-null   object
 5   content       2063 non-null   object
 6   summary       2063 non-null   object
dtypes: object(7)
memory usage: 112.9+ KB


In [ ]:
combined_df.head()

,title,category,publish_date,author,article_url,content,summary
0,Pentingnya Hak Cipta dan Perlakuan Maksimal un...,detikHot,2024-07-01,Mauludi Rismoyo,https://hot.detik.com/music/d-7417986/pentingn...,Perseteruan Ahmad Dhani dan Once beberapa wakt...,Perseteruan Ahmad Dhani dan Once beberapa wakt...
1,Okan Kornelius Pakai Kemeja Pink dan Dapat Kej...,detikHot,2024-07-01,prih febriani,https://hot.detik.com/celeb/d-7417696/okan-kor...,Artis Okan Kornelius saat ini diketahui tengah...,Artis Okan Kornelius saat ini diketahui tengah...
2,Ayu Ting Ting Putus dengan Muhammad Fardhana,detikHot,2024-07-01,Muhammad Ahsan Nurrijal,https://hot.detik.com/celeb/d-7418047/ayu-ting...,Pedangdut Ayu Ting Ting rupanya sudah tak menj...,Pedangdut Ayu Ting Ting rupanya sudah tak menj...
3,Dion Wiyoko Ajak Anak Muda Tak Takut Berkeringat,detikHot,2024-07-01,Dicky Ardian,https://hot.detik.com/hottainment/d-7417905/di...,Siapa sangka aktor Dion Wiyoko dulunya jago ol...,Siapa sangka aktor Dion Wiyoko dulunya jago ol...
4,Denny Sumargo Ungkap Kondisi Janin di Kandunga...,detikHot,2024-07-01,Febryantino Nur Pratama,https://hot.detik.com/celeb/d-7416625/denny-su...,"Aktor dan mantan atlet basket, Denny Sumargo b...","Aktor dan mantan atlet basket, Denny Sumargo b..."


In [ ]:
nan_summary = combined_df[combined_df['summary'].isna()]


nan_summary

,title,category,publish_date,author,article_url,content,summary


In [ ]:
combined_df=combined_df.dropna()

# Duplicate


In [ ]:
data = combined_df.copy()

In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2063 entries, 0 to 2062
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   title         2063 non-null   object
 1   category      2063 non-null   object
 2   publish_date  2063 non-null   object
 3   author        2063 non-null   object
 4   article_url   2063 non-null   object
 5   content       2063 non-null   object
 6   summary       2063 non-null   object
dtypes: object(7)
memory usage: 112.9+ KB


# Tokenize

In [ ]:
# tokenizer = AutoTokenizer.from_pretrained("indobenchmark/indobert-base-p2", use_fast=True)
# tokenizer.add_tokens([ 'NaN' ])

In [ ]:
# def tokenize_function(text):
#     return tokenizer(text, padding='max_length', max_length=256, truncation=True)

# # Tokenisasi kolom 'title' dan 'summary'
# data['tokenized_title'] = data['title'].apply(lambda x: tokenize_function(x))
# data['tokenized_summary'] = data['summary'].apply(lambda x: tokenize_function(x))

In [ ]:
# data.head()

In [ ]:
# # Menampilkan hasil tokenisasi untuk 'tokenized_title' dan 'tokenized_content'
# print("Tokenized Title:")
# print(data[['title', 'tokenized_title']])

# print("\nTokenized summary:")
# print(data[['summary', 'tokenized_summary']])

# # Menampilkan hanya input_ids yang lebih mudah dibaca
# print("\nInput IDs for Tokenized Title:")
# print(data['tokenized_title'].apply(lambda x: x['input_ids']))

# print("\nInput IDs for Tokenized summary:")
# print(data['tokenized_summary'].apply(lambda x: x['input_ids']))

Mencoba kode lain agar tokenize lebih mudah di pahami

# Word Embedding

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

# Memuat IndoBERT dan tokenizer
tokenizer = AutoTokenizer.from_pretrained("indobenchmark/indobert-base-p1", use_fast=True)
model = AutoModel.from_pretrained("indobenchmark/indobert-base-p1")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

In [ ]:
# Fungsi untuk mendapatkan embedding dari IndoBERT
def get_embeddings(input_ids):
    with torch.no_grad():
        outputs = model(input_ids=input_ids)
    # Mengambil hidden states dari layer terakhir dan rata-rata embeddingsnya
    embeddings = outputs.last_hidden_state.mean(dim=1)
    return embeddings

# Fungsi untuk mengonversi tokenized data ke dalam bentuk input_ids dan mendapatkan embedding
def get_input_ids_and_embeddings(data_column):
    input_ids_list = []
    embeddings_list = []

    for text in data_column:
        # Tokenisasi
        tokens = tokenizer(text, padding='max_length', max_length=256, truncation=True, return_tensors="pt")
        input_ids = tokens['input_ids']  # Ambil input_ids
        input_ids_list.append(input_ids)

        # Mendapatkan embeddings
        embedding = get_embeddings(input_ids)
        embeddings_list.append(embedding)

    return input_ids_list, embeddings_list

# Dapatkan input_ids dan embeddings untuk title dan summary
title_input_ids, title_embeddings = get_input_ids_and_embeddings(data['title'])
summary_input_ids, summary_embeddings = get_input_ids_and_embeddings(data['summary'])


We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.


## Mengubah embeddings menjadi array numpy

In [ ]:
# Mengubah embeddings menjadi array numpy agar bisa disimpan dalam DataFrame
title_embeddings_array = np.vstack([embedding.numpy() for embedding in title_embeddings])
summary_embeddings_array = np.vstack([embedding.numpy() for embedding in summary_embeddings])

# Membuat DataFrame baru dengan embeddings numerik
data_embeddings = pd.DataFrame({
    'title_embeddings': list(title_embeddings_array),
    'summary_embeddings': list(summary_embeddings_array)
})

# Menyimpan DataFrame embeddings ke dalam file CSV (atau format lain)
# data_embeddings.to_csv('/content/drive/MyDrive/File/CODES/Similarity/Detik Berita/detik_berita_embeddings.csv', index=False)


# Cosine Similarity

In [ ]:
# hitung cosine similarity pada matrix berita
cosine_sim = cosine_similarity(title_embeddings_array, summary_embeddings_array)
print(cosine_sim)

[[0.4200536  0.5428125  0.364487   ... 0.8739842  0.53526455 0.76630294]
 [0.42167103 0.54701763 0.3654223  ... 0.8803202  0.53826904 0.7710594 ]
 [0.4178853  0.5411078  0.3642066  ... 0.87328076 0.5335995  0.76440597]
 ...
 [0.41769448 0.5417557  0.36280203 ... 0.87655383 0.5346513  0.76789296]
 [0.4185007  0.5411456  0.36328658 ... 0.8742609  0.5352633  0.76581323]
 [0.41965622 0.54442537 0.36279303 ... 0.8804164  0.536415   0.7713593 ]]


In [ ]:
# membuat dataframe dari varible cosine similarity dengan baris dan kolom title
cosine_sim_df = pd.DataFrame(cosine_sim, index=data['summary'], columns=data['title'])

print(cosine_sim_df.shape)

(2063, 2063)


## Check Similarity Result

In [ ]:
# Menampilkan pasangan similarity antara 'title' dan 'summary' yang sesuai dengan urutan
for index, row in cosine_sim_df.iterrows():
    # The index is a summary, and you need to find the corresponding title
    # to access the similarity value. However, since you have both summaries
    # and titles as indices, you can access the similarity directly using
    # the index and the corresponding title from the 'data' DataFrame.

    title = data.loc[data['summary'] == index, 'title'].iloc[0] # Get the title for the current summary
    similarity_value = cosine_sim_df.loc[index, title]  # Access the similarity using both summary and title

    print(f"Title: {title}")
    print(f"Summary: {index}")
    print(f"Cosine Similarity: {similarity_value}\n")
    print("-" * 50)  # Pemisah antar pasangan

Streaming output truncated to the last 5000 lines.
Title: Thalita Latief Nyaman dengan Cara Kekasih Dekati Anak
Summary: Artis Thalita Latief menceritakan kekasihnya, Ichsan Reinaldy sudah sangat dekat dengan putra semata wayangnya. Surprisingly atau mungkin karena dia juga punya banyak ponakan kali. Saya nggak pernah kayak gitu," kata Thalita Latief ditemui di Studio Trans TV, Jakarta Selatan, kemarin. Di sisi lain, Thalita menilai sosok Ichsan pantang menyerah membangun kedekatan dengan putranya. Bahkan mantan istri Dennis Lyla itu menyebut putranya sangat menerima sosok Ichsan. "Dia juga effortless, benar-benar apa adanya aja. Pasangan aku bisa support hobi dia (anak). Pasangan bisa dekat dengan anak menjadi hal penting untuk Thalita. " Sekarang dapat deh tuh ibaratnya gengnya. Jadi ya sudah selama pasangan ini suka dan enjoy tidak terganggu ya silakan aja yang penting ya aku penginnya Rafello merasa nyaman," tukas Thalita Latief.
Cosine Similarity: 0.4998360872268677

-------------

In [ ]:
# List to store the results
results = []

# Menampilkan pasangan similarity antara 'title' dan 'summary' yang sesuai dengan urutan
for index, row in cosine_sim_df.iterrows():
    # Mendapatkan judul yang sesuai dengan summary berdasarkan index
    title = data.loc[data['summary'] == index, 'title'].iloc[0]
    similarity_value = cosine_sim_df.loc[index, title]  # Mengakses nilai similarity

    # Menyimpan pasangan dan nilai similarity ke dalam list
    results.append([title, index, similarity_value])

# Membuat DataFrame untuk hasilnya
similarity_df = pd.DataFrame(results, columns=['Title', 'Summary', 'Cosine Similarity'])


In [ ]:
similarity_df.head()

,Title,Summary,Cosine Similarity
0,Pentingnya Hak Cipta dan Perlakuan Maksimal un...,Perseteruan Ahmad Dhani dan Once beberapa wakt...,0.420054
1,Okan Kornelius Pakai Kemeja Pink dan Dapat Kej...,Artis Okan Kornelius saat ini diketahui tengah...,0.547018
2,Ayu Ting Ting Putus dengan Muhammad Fardhana,Pedangdut Ayu Ting Ting rupanya sudah tak menj...,0.364207
3,Dion Wiyoko Ajak Anak Muda Tak Takut Berkeringat,Siapa sangka aktor Dion Wiyoko dulunya jago ol...,0.911709
4,Denny Sumargo Ungkap Kondisi Janin di Kandunga...,"Aktor dan mantan atlet basket, Denny Sumargo b...",0.505139


## Save Similarity

In [ ]:
# Menyimpan DataFrame ke file CSV di lokasi yang ditentukan
similarity_df.to_csv('/content/drive/MyDrive/File/CODES/Similarity/2_Similarity_Detik_Hot_Base1/similarity_detik_hot_base1.csv', index=False)